In [1]:
from mhctools import NetMHC, NetMHC3, NetMHC4
from mhcnames import normalize_allele_name## ONLY FOR LINUX

In [3]:
# For now we should define single organism and single class
CLASS_TO_USE = ['I'
ORGANISMS_TO_USE = ['Mouse']


PER_ORGANISM_PER_CLASS_ALLELES = {
    'Human': {
        'I': ['HLA_A_02_01','HLA_B_27_05', "HLA_A_03_01", "HLA_B_07_02"],
        'II': ['HLA_DR', 'HLA_DRB1_01_01', 'HLA_DRB1_04_01', 'HLA_DQ']
    },
    'Mouse': {
        'I':['H2_Db','H2_Kb'],
        'II': ['H2_IAb', 'H2_IAd', 'H2_d class II', 'H2_b class II']
    }
    
}

## Mouse
ALLELES = PER_ORGANISM_PER_CLASS_ALLELES[ORGANISMS_TO_USE][CLASS_TO_USE]

DECOY_SUFFIX = ORGANISMS_TO_USE + "_" + CLASS_TO_USE
binders_filename_base = "/home/denklewer/Data/val_data/input_dataset_binders_val"
DECOY_PATH = "/home/denklewer/Data/val_data/input_dataset_decoy_val_" + DECOY_SUFFIX + ".fasta"


In [4]:
def readFasta(filename): 
    fasta = {}
    with open(filename) as file_one:
        for line in file_one:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                active_sequence_name = line[1:]
                continue
            sequence = line
            fasta[active_sequence_name] = sequence
    return fasta


In [5]:
per_allele_fasta = {}
per_allele_decoy_fasta = {}


filename = DECOY_PATH
decoy_fasta = readFasta(filename)

for allele_name in ALLELES:
    filename = binders_filename_base + "_" + allele_name + ".fasta"
    fasta = readFasta(filename)
    per_allele_fasta[allele_name] = fasta
    per_allele_decoy_fasta[allele_name] = dict(list(decoy_fasta.items())[:len(fasta)])




print([(len(per_allele_fasta[allele_name]), len(per_allele_decoy_fasta[allele_name])) for allele_name in ALLELES])

FileNotFoundError: [Errno 2] No such file or directory: '/home/denklewer/Data/val_data/input_dataset_decoy_val_Mouse_I.fasta'

In [9]:
def splitByLengthAndFilter(fasta_dict, min_length = 8):
    unique_lengths = list(set(map(len, list(fasta_dict.values()))))
    unique_lengths = list(filter(lambda x: x>=min_length, unique_lengths ))
    per_length_dict_of_dicts = {target_length :{k: v for k, v in fasta_dict.items() if len(v) == target_length} for target_length in unique_lengths}
    return per_length_dict_of_dicts



In [10]:
per_allele_per_length_dict = dict() 
per_allele_per_length_decoy_dict = dict()

for allele_name in ALLELES:
    fasta_dict = per_allele_fasta[allele_name]
    per_length_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
    print(f"allele fasta splited into {len(per_length_dict_of_dicts)} dicts")
    per_allele_per_length_dict[allele_name] = per_length_dict_of_dicts
    
        
    ## make same for decoy dicts
    fasta_dict = per_allele_decoy_fasta[allele_name]
    per_length_decoy_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
    print(f"decoy fasta splited into {len(per_length_decoy_dict_of_dicts)} dicts")
    per_allele_per_length_decoy_dict[allele_name] = per_length_decoy_dict_of_dicts


# fasta_dict = decoy_fasta
# per_length_decoy_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
# print(f"splited into {len(per_length_decoy_dict_of_dicts)} dicts")




allele fasta splited into 13 dicts
decoy fasta splited into 5 dicts
allele fasta splited into 14 dicts
decoy fasta splited into 5 dicts


In [11]:
per_allele_per_length_dict[allele_name].keys()

dict_keys([8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21])

In [12]:
ALLELES = list(per_allele_per_length_dict.keys())
ALLELES

['H2_Db', 'H2_Kb']

In [13]:
normalize_allele_name(ALLELES[0])

'H-2-Db'

In [14]:
%%time
def predictSequences(dict_of_dicts, allele_name):
    alleles = [normalize_allele_name(allele_name)]
    predictor = NetMHC4(
            alleles=alleles,
            program_name='netMHC')
    print(f"Now we are going to predict binders for  dataset {allele_name} using alleles parameter {alleles}")
    binding_predictions = [predictor.predict_subsequences(fasta_dict, 
                                                          peptide_lengths=[length]) # peek any element and take length
                           for length, fasta_dict in dict_of_dicts.items()]
    lengths = [len(prediction) for prediction in binding_predictions] 
    print(f"total processed peptides after filtering -  {sum(lengths)}")
    return binding_predictions



CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 7.87 µs


In [307]:
per_allele_binder_predictions = dict()
per_allele_non_binder_predictions = dict()
for allele_name, per_length_dict in per_allele_per_length_dict.items():
    predictions = predictSequences(per_length_dict, allele_name)
    per_allele_binder_predictions[allele_name] = predictions
    print("do same for decoy")
    predictions = predictSequences(per_allele_per_length_decoy_dict[allele_name], allele_name)
    per_allele_non_binder_predictions[allele_name] = predictions
    
    
    


Now we are going to predict binders for  dataset H2_Db using alleles parameter ['H-2-Db']
total processed peptides after filtering -  618
do same for decoy
Now we are going to predict binders for  dataset H2_Db using alleles parameter ['H-2-Db']
total processed peptides after filtering -  618
Now we are going to predict binders for  dataset H2_Kb using alleles parameter ['H-2-Kb']
total processed peptides after filtering -  806
do same for decoy
Now we are going to predict binders for  dataset H2_Kb using alleles parameter ['H-2-Kb']
total processed peptides after filtering -  806


### Join predictions for given allele

#### join by length

In [308]:
per_allele_total_binders_predictions = dict()
per_allele_total_non_binders_predictions = dict()
for allele_name, predictions in per_allele_binder_predictions.items():
    list_of_df = [prediction.to_dataframe() for prediction in predictions]
    df_binders_result = pd.concat(list_of_df, ignore_index=True)
    df_binders_result['binder'] = 1
    per_allele_total_binders_predictions[allele_name] = df_binders_result
    
for allele_name, predictions in per_allele_non_binder_predictions.items():
    list_of_df = [prediction.to_dataframe() for prediction in predictions]
    df_non_binders_result = pd.concat(list_of_df, ignore_index=True)
    df_non_binders_result['binder'] = 0
    per_allele_total_non_binders_predictions[allele_name] = df_non_binders_result
    
    
    
    

#### join binders/non_binders

In [309]:
per_allele_total_predictions = dict()
for allele_name in per_allele_binder_predictions.keys():
    df1 = per_allele_total_binders_predictions[allele_name]
    df2 = per_allele_total_non_binders_predictions[allele_name]
    result_df =  pd.concat([df1, df2], ignore_index=True)
    per_allele_total_predictions[allele_name] = result_df

In [310]:
for allele_name, total_df in per_allele_total_predictions.items():
    total_df.to_csv(f"/home/denklewer/data//mhc_predictions_{allele_name}")

In [312]:
per_allele_total_predictions[ALLELES[0]]

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length,binder
0,sec415,0,AAFRYTLL,H-2-Db,0.156,9281.19,2.5,netMHC,8,1
1,sec95,0,AAYGFRNI,H-2-Db,0.084,20161.57,7.0,netMHC,8,1
2,sec300,0,AAYSFYNV,H-2-Db,0.134,11705.29,3.5,netMHC,8,1
3,sec524,0,ANLKYLSL,H-2-Db,0.045,30595.03,15.0,netMHC,8,1
4,sec398,0,ASKPTLNE,H-2-Db,0.005,47622.84,75.0,netMHC,8,1
...,...,...,...,...,...,...,...,...,...,...
1231,sec250,0,WMTYRQHRDAEE,H-2-Db,0.008,45708.48,55.0,netMHC,12,0
1232,sec49,0,WSCTEARCPGIC,H-2-Db,0.020,40288.71,30.0,netMHC,12,0
1233,sec360,0,YCDHKEINHLFC,H-2-Db,0.015,42392.77,37.0,netMHC,12,0
1234,sec106,0,YGGMNLTMNLTA,H-2-Db,0.239,3762.77,1.0,netMHC,12,0
